# Clasificador de huevos: `good` vs `crack`

Notebook educativo para Google Colab con GPU. Integra preprocesamiento digital, una red densa (MLP) y una red convolucional (CNN). El objetivo principal es minimizar los falsos negativos de la clase `crack`: un huevo agrietado clasificado como bueno es el error más costoso.

## Ruta de trabajo
1. Configurar Colab y GPU.
2. Descargar el dataset COCO desde Roboflow.
3. Recortar y estandarizar los huevos.
4. Entrenar MLP y CNN.
5. Comparar desempeño y exportar el mejor modelo a Drive.

La API key se solicitará de forma oculta y no se guardará en el notebook.

In [ ]:
# Configuración base. Ejecutar en Google Colab.
!pip -q install roboflow

import os
import json
import time
import shutil
import random
import warnings
from pathlib import Path
from getpass import getpass

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from google.colab import drive
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, recall_score

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
IMG_SIZE = (100, 100)
BATCH_SIZE = 32
CLASS_NAMES = ['good', 'crack']
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Verificación explícita de GPU.
!nvidia-smi

In [ ]:
# PEGA AQUÍ TU API KEY (se oculta y no queda escrita en el archivo).
# Se obtiene en: https://app.roboflow.com/settings/api
API_KEY = getpass('API key de Roboflow: ')
if not API_KEY.strip():
    raise ValueError('La API key no puede estar vacía.')
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/egg_quality')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

## 1. Descarga del dataset
Roboflow entrega las imágenes y las anotaciones COCO. Trabajaremos con las cajas delimitadoras para demostrar el recorte y la estandarización antes de entrenar.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)
project = rf.workspace('muhammad-fauzan-wbvuk').project('egg-pisqc')
versions = list(project.versions())
print('Versiones disponibles:', versions)
if not versions:
    raise RuntimeError('Roboflow no devolvió versiones para el proyecto.')
version_number = 1 if any(getattr(v, 'version', None) == 1 for v in versions) else getattr(versions[0], 'version', 1)
version = project.version(version_number)
dataset = version.download('coco')
DATASET_DIR = Path(dataset.location)
print('Dataset:', DATASET_DIR)
for path in sorted(DATASET_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(DATASET_DIR))

## 2. Preprocesamiento por bounding box

Las imágenes se recortan alrededor del huevo, se añade un margen pequeño y se llevan a `100x100` píxeles. Esto reduce el costo computacional, elimina fondo irrelevante y entrega el mismo formato a ambos modelos. La normalización `/255` se hará al cargar los crops.

In [ ]:
def find_annotation_file(split_dir):
    files = list(split_dir.glob('_annotations.coco.json'))
    return files[0] if files else None

def find_image(images_dir, file_name):
    candidate = images_dir / file_name
    if candidate.exists():
        return candidate
    matches = list(images_dir.rglob(Path(file_name).name))
    return matches[0] if matches else None

def crop_from_bbox(image, bbox, margin=0.04):
    x, y, width, height = [float(value) for value in bbox]
    image_height, image_width = image.shape[:2]
    margin_x, margin_y = width * margin, height * margin
    x1 = max(0, int(x - margin_x))
    y1 = max(0, int(y - margin_y))
    x2 = min(image_width, int(x + width + margin_x))
    y2 = min(image_height, int(y + height + margin_y))
    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    return cv2.resize(crop, IMG_SIZE, interpolation=cv2.INTER_AREA)

def process_split(split_name):
    split_dir = DATASET_DIR / split_name
    if not split_dir.exists():
        return 0
    annotation_path = find_annotation_file(split_dir)
    if annotation_path is None:
        print('Sin anotaciones COCO:', split_name)
        return 0
    with annotation_path.open(encoding='utf-8') as file:
        coco = json.load(file)
    print(split_name, '->', len(coco.get('images', [])), 'imágenes y', len(coco.get('annotations', [])), 'anotaciones')
    categories = {item['id']: item['name'].lower() for item in coco.get('categories', [])}
    images = {item['id']: item for item in coco.get('images', [])}
    output_count = 0
    for annotation in coco.get('annotations', []):
        label = categories.get(annotation['category_id'], '').lower()
        label = 'crack' if 'crack' in label else 'good' if 'good' in label else None
        image_info = images.get(annotation['image_id'])
        if label is None or image_info is None:
            continue
        image_path = find_image(split_dir, image_info['file_name'])
        if image_path is None:
            continue
        image = cv2.imread(str(image_path))
        crop = crop_from_bbox(image, annotation['bbox'])
        if crop is None:
            continue
        output_dir = Path('/content/crops') / split_name / label
        output_dir.mkdir(parents=True, exist_ok=True)
        output_path = output_dir / f"{Path(image_info['file_name']).stem}_{annotation['id']}.jpg"
        cv2.imwrite(str(output_path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 95])
        output_count += 1
    return output_count

counts = {split: process_split(split) for split in ['train', 'valid', 'test']}
print('Crops generados:', counts)

In [ ]:
# Inspección visual: bbox original, recorte, normalización y Sobel.
def first_example(split='train', label='crack'):
    split_dir = DATASET_DIR / split
    annotation_path = find_annotation_file(split_dir)
    with annotation_path.open(encoding='utf-8') as file:
        coco = json.load(file)
    categories = {item['id']: item['name'].lower() for item in coco['categories']}
    images = {item['id']: item for item in coco['images']}
    for annotation in coco['annotations']:
        current = categories.get(annotation['category_id'], '')
        if label in current:
            info = images[annotation['image_id']]
            path = find_image(split_dir, info['file_name'])
            image_bgr = cv2.imread(str(path))
            image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
            x, y, width, height = map(int, annotation['bbox'])
            boxed = image_rgb.copy()
            cv2.rectangle(boxed, (x, y), (x + width, y + height), (255, 0, 0), 3)
            crop = crop_from_bbox(image_bgr, annotation['bbox'])
            gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
            sobel = cv2.filter2D(gray, -1, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32))
            return boxed, crop, crop.astype(np.float32) / 255.0, sobel
    return None

example = first_example()
if example:
    titles = ['Original con bbox', 'Recorte 100x100', 'Normalizado /255', 'Filtro Sobel']
    plt.figure(figsize=(14, 4))
    for index, (title, image) in enumerate(zip(titles, example), 1):
        plt.subplot(1, 4, index)
        plt.imshow(image, cmap='gray' if index == 4 else None)
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Mosaico de ejemplos y balance por split.
def show_mosaic(split='train', per_class=25):
    plt.figure(figsize=(10, 4))
    for row, label in enumerate(CLASS_NAMES):
        files = list((Path('/content/crops') / split / label).glob('*.jpg'))[:per_class]
        for column, path in enumerate(files):
            image = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
            axis = plt.subplot(2, per_class, row * per_class + column + 1)
            axis.imshow(image)
            axis.axis('off')
            if column == 0:
                axis.set_ylabel(label)
    plt.suptitle(f'Mosaico: {split}')
    plt.tight_layout()
    plt.show()

show_mosaic()
balance = []
for split in ['train', 'valid', 'test']:
    for label in CLASS_NAMES:
        balance.append({'split': split, 'clase': label, 'imagenes': len(list((Path('/content/crops') / split / label).glob('*.jpg')))})
balance_df = pd.DataFrame(balance)
display(balance_df.pivot(index='split', columns='clase', values='imagenes'))
print('Si hay un desbalance importante, se puede usar class_weight o augmentation dirigida a la clase minoritaria.')

## 3. Carga, normalización y augmentation
`image_dataset_from_directory` produce etiquetas enteras, por eso usaremos `sparse_categorical_crossentropy`. `prefetch` solapa la preparación de lotes con el entrenamiento. La augmentation agrega variaciones pequeñas para combatir el sobreajuste con un dataset reducido.

In [ ]:
def load_split(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        Path('/content/crops') / split,
        labels='inferred',
        label_mode='int',
        class_names=CLASS_NAMES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

train_raw = load_split('train', True)
valid_raw = load_split('valid', False)
test_raw = load_split('test', False)

normalization = tf.keras.layers.Rescaling(1.0 / 255)
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
], name='augmentation')
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_raw.map(lambda images, labels: (normalization(images), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
valid_ds = valid_raw.map(lambda images, labels: (normalization(images), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_raw.map(lambda images, labels: (normalization(images), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## 4. Modelo MLP
`Flatten` convierte cada imagen en un vector. Las capas densas aprenden combinaciones globales de píxeles; ReLU se usa en las capas ocultas y softmax entrega una probabilidad por clase.

## 5. Modelo CNN
Las convoluciones detectan bordes y patrones locales reutilizando filtros. MaxPooling reduce la resolución conservando señales dominantes. Dropout apaga unidades durante el entrenamiento y ayuda a limitar el sobreajuste.

In [ ]:
def callbacks_for(path):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6),
        tf.keras.callbacks.ModelCheckpoint(path, monitor='val_loss', save_best_only=True),
    ]

cnn = tf.keras.Sequential([
    tf.keras.Input(shape=(*IMG_SIZE, 3)),
    augmentation,
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(2, activation='softmax'),
], name='cnn')
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()
start = time.perf_counter()
history_cnn = cnn.fit(train_ds, validation_data=valid_ds, epochs=50, callbacks=callbacks_for('/content/mejor_cnn.keras'))
cnn_time = time.perf_counter() - start

In [ ]:
# La evaluación comparativa comienza en la celda siguiente.
# Se revisan accuracy y recall de la clase crack.

In [ ]:
def plot_history(history, title):
    plt.plot(history.history['loss'], label='loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.plot(history.history['accuracy'], label='accuracy')
    plt.plot(history.history['val_accuracy'], label='val_accuracy')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plot_history(history_mlp, 'MLP')
plt.subplot(1, 2, 2)
plot_history(history_cnn, 'CNN')
plt.tight_layout()
plt.show()

def evaluate_model(model, name, elapsed):
    loss, accuracy = model.evaluate(test_ds, verbose=0)
    y_true = np.concatenate([labels.numpy() for _, labels in test_ds])
    probabilities = model.predict(test_ds, verbose=0)
    y_pred = probabilities.argmax(axis=1)
    print(name, '| loss:', round(loss, 4), '| accuracy:', round(accuracy, 4))
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASS_NAMES).plot(cmap='Blues')
    plt.title(f'Matriz de confusión: {name}')
    plt.show()
    return {
        'modelo': name,
        'accuracy': accuracy,
        'recall_crack': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'params': model.count_params(),
        'tiempo_seg': elapsed,
    }, y_true, y_pred, probabilities

mlp_result = evaluate_model(mlp, 'MLP', mlp_time)
cnn_result = evaluate_model(cnn, 'CNN', cnn_time)
comparison = pd.DataFrame([mlp_result[0], cnn_result[0]])
display(comparison)

In [ ]:
best_model = cnn if cnn_result[0]['recall_crack'] >= mlp_result[0]['recall_crack'] else mlp
best_name = 'CNN' if best_model is cnn else 'MLP'
images, labels = next(iter(test_ds))
predictions = best_model.predict(images, verbose=0)
plt.figure(figsize=(15, 6))
for index in range(min(10, len(images))):
    predicted = int(predictions[index].argmax())
    actual = int(labels[index])
    color = 'green' if predicted == actual else 'red'
    plt.subplot(2, 5, index + 1)
    plt.imshow(images[index])
    plt.title(f'Real: {CLASS_NAMES[actual]}\nPred: {CLASS_NAMES[predicted]} ({predictions[index][predicted]:.1%})', color=color, fontsize=9)
    plt.axis('off')
plt.suptitle(f'Predicciones de {best_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Persistencia y contrato para la VM: RGB, recorte, 100x100 y /255.

In [ ]:
mlp = tf.keras.Sequential([
    tf.keras.Input(shape=(*IMG_SIZE, 3)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax'),
], name='mlp')
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()
start = time.perf_counter()
history_mlp = mlp.fit(train_ds, validation_data=valid_ds, epochs=50, callbacks=callbacks_for('/content/mejor_mlp.keras'))
mlp_time = time.perf_counter() - start

In [ ]:
best_path = '/content/mejor_cnn.keras' if best_model is cnn else '/content/mejor_mlp.keras'
export_dir = DRIVE_DIR / 'models'
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_path, export_dir / Path(best_path).name)
with (export_dir / 'clases.json').open('w', encoding='utf-8') as file:
    json.dump(CLASS_NAMES, file, ensure_ascii=False, indent=2)
preprocessing = {
    'crop_size': [100, 100],
    'scale': '/255',
    'color_format': 'RGB',
    'bbox_margin': 0.04,
}
with (export_dir / 'preprocesamiento.json').open('w', encoding='utf-8') as file:
    json.dump(preprocessing, file, ensure_ascii=False, indent=2)
zip_base = '/content/egg_quality_models'
zip_path = shutil.make_archive(zip_base, 'zip', export_dir)
shutil.copy2(zip_path, DRIVE_DIR / 'egg_quality_models.zip')
print('Modelo:', export_dir / Path(best_path).name)
print('ZIP:', DRIVE_DIR / 'egg_quality_models.zip')
print('Existe el ZIP:', (DRIVE_DIR / 'egg_quality_models.zip').exists())

In [ ]:
# Para la VM: descargar egg_quality_models.zip, descomprimirlo y cargar el modelo con tf.keras.models.load_model().